

## RAG-Based Nutritional Chat-Bot

A nutritional chatbot based on retrieval augmented pipeline.




In [4]:
import os

if "COLAB_GPU" in os.environ:
  print("[INFO] Running google Colab, installing requirements")
  !pip install PyMuPDF #for reading PDFS
  !pip install tqdm  #for progress bars
  !pip install accelerate
  !pip install bitsandbytes
  !pip install flash-attn --no-build-isolation

[INFO] Running google Colab, installing requirements
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 40.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=255984554 sha256=51f6422861ed951b968428adc9fa7406027f73f2145be5e163810df6f459abea
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash-attn


In [1]:
!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U transformers sentence-transformers

Found existing installation: torch 2.8.0+cu126
Uninstalling torch-2.8.0+cu126:
  Successfully uninstalled torch-2.8.0+cu126
Found existing installation: torchvision 0.23.0+cu126
Uninstalling torchvision-0.23.0+cu126:
  Successfully uninstalled torchvision-0.23.0+cu126
Found existing installation: torchaudio 2.8.0+cu126
Uninstalling torchaudio-2.8.0+cu126:
  Successfully uninstalled torchaudio-2.8.0+cu126
Found existing installation: transformers 4.57.1
Uninstalling transformers-4.57.1:
  Successfully uninstalled transformers-4.57.1
Found existing installation: sentence-transformers 5.1.2
Uninstalling sentence-transformers-5.1.2:
  Successfully uninstalled sentence-transformers-5.1.2
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 106.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 106.4 MB/s eta 0:00:00
     ━━━

## Document Processing/ Text Processing and Embedding Creation


In [2]:
import os
import requests

pdf_path = "/content/Human-Nutrition.pdf"

if not os.path.exists(pdf_path):
  print("File doesn't exist, downloading...")
  url= "https://pressbooks.oer.hawaii.edu/humannutrition2/"
  filename=pdf_path
  response = requests.get(url)

  if response.status_code == 200:
    with open(filename, "wb") as file:
      file.write(response.content)
    print(f"The file has been downloaded and saved as {filename}")
  else:
    print(f"Failed to download: {response.statu_code}")
else:
  print(f"File {pdf_path} exists")


File doesn't exist, downloading...
The file has been downloaded and saved as /content/Human-Nutrition.pdf


In [5]:
import fitz #
from tqdm.auto import tqdm

def text_formatter(text: str) -> str:
  """ Perform minor formatting on text"""
  cleaned_text = text.replace("\n", " ").strip()
  return cleaned_text


def open_read_pdf( pdf_path: str) -> list[dict]:
  """ Opens PDF file read its conents
      Parameters: pdf_path(str):
      Returns: list[dict] a list of dictionaries each containing page number
  """
  doc = fitz.open(pdf_path) #open a document
  pages_and_texts = []
  for page_number, page in tqdm(enumerate(doc)):
    text= page.get_text()
    text= text_formatter(text) #remove empty spaces
    pages_and_texts.append({"page_number": page_number -41,
                            "page_char_count": len(text),
                            "page_word_count": len(text.split(" ")),
                            "page_sentence_count_raw": len(text.split(". ")),
                            "page_token_count": len(text) / 4,
                            "text": text})
  return pages_and_texts
pages_and_texts = open_read_pdf(pdf_path= pdf_path)
pages_and_texts[:2]


0it [00:00, ?it/s]

[{'page_number': -41,
  'page_char_count': 93,
  'page_word_count': 22,
  'page_sentence_count_raw': 1,
  'page_token_count': 23.25,
  'text': 'Skip to content [image] Menu Primary Navigation •  Home •  Read •  Sign in •  Search in book:'},
 {'page_number': -40,
  'page_char_count': 145,
  'page_word_count': 23,
  'page_sentence_count_raw': 2,
  'page_token_count': 36.25,
  'text': 'Search Want to create or adapt books like this? Learn more about how Pressbooks supports open publishing practices.  Book Title: Human Nutrition:'}]

In [6]:
import random
random.sample(pages_and_texts, k=3)

[{'page_number': -10,
  'page_char_count': 22,
  'page_word_count': 3,
  'page_sentence_count_raw': 1,
  'page_token_count': 5.5,
  'text': 'Pressbooks on LinkedIn'},
 {'page_number': -15,
  'page_char_count': 106,
  'page_word_count': 16,
  'page_sentence_count_raw': 1,
  'page_token_count': 26.5,
  'text': 'Metadata Title Human Nutrition: 2020 Edition Authors University of Hawai‘i at Mānoa Food Science and Human'},
 {'page_number': -21,
  'page_char_count': 856,
  'page_word_count': 129,
  'page_sentence_count_raw': 11,
  'page_token_count': 214.0,
  'text': '1. Introduction University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 2. Childhood University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 3. Adolescence University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 4. Late Adolescence University of Hawai‘i at Mānoa Food Science and Human Nutrition Prog

In [7]:
import pandas as pd

df= pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-41,93,22,1,23.25,Skip to content [image] Menu Primary Navigatio...
1,-40,145,23,2,36.25,Search Want to create or adapt books like this...
2,-39,552,90,3,138.00,2020 Edition by University of Hawai‘i at Mānoa...
3,-38,0,1,1,0.00,
4,-37,83,12,1,20.75,Creative Commons Attribution Read Book Content...


In [8]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,32.00,32.00,32.00,32.00,32.00
mean,-25.50,616.06,93.53,7.09,154.02
std,9.38,370.73,55.86,4.63,92.68
min,-41.00,0.00,1.00,1.00,0.00
25%,-33.25,135.25,22.75,1.75,33.81
50%,-25.50,854.50,128.50,10.00,213.62
75%,-17.75,906.25,135.50,11.00,226.56
max,-10.00,954.00,149.00,14.00,238.50


Chunking -  

5 types of chunking
1. Fixedsize chunking
2. Sematic chunking
3. Recursive chunking
4. Structural chunking
5. LLM chunking

Further text processing (splitting pages into sentences)- Fixed size chunking

In [9]:
from spacy.lang.en import English
nlp= English()

nlp.add_pipe("sentencizer")
doc= nlp("This is a sentence. This is another sentence.")
assert len(list(doc.sents))== 2

list(doc.sents)




[This is a sentence., This is another sentence.]

In [10]:
for item in tqdm(pages_and_texts):
  item["sentences"] = list(nlp(item["text"]).sents)
  #all sentences are string
  item["sentences"] = [str(sentence) for sentence in item["sentences"]]
  # count the sentences
  item["page_sentence_count_spacy"] = len(item["sentences"])

  0%|          | 0/32 [00:00<?, ?it/s]

In [11]:
random.sample(pages_and_texts, k=1)

[{'page_number': -28,
  'page_char_count': 861,
  'page_word_count': 131,
  'page_sentence_count_raw': 14,
  'page_token_count': 215.25,
  'text': '7. Proteins in a Nutshell University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 8. Proteins, Diet, and Personal Choices University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 10. VII. Chapter 7. Alcohol 1. Introduction University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 2. Alcohol Metabolism University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 3. Health Consequences of Alcohol Abuse University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 4. Health Benefits of Moderate Alcohol Intake University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 11. VIII. Chapter 8. Energy 1. Int

Chunking our sentences together

In [12]:
num_sentence_chunk_size = 10

def split_list(input_list, slice_size: int) -> list[list[str]]:
  """
  splits the input_list into sublists of size slice_size

  """
  return[input_list[i:i+ slice_size] for i in range(0, len(input_list), slice_size)]

#loop through pages and texts

for item in tqdm(pages_and_texts):
  item["sentences_chunks"] = split_list(input_list= item["sentences"], slice_size= num_sentence_chunk_size)

  item["num_chunks"] = len(item["sentences_chunks"])

  0%|          | 0/32 [00:00<?, ?it/s]

In [15]:
df_chunks = pd.DataFrame(pages_and_chunks)
df_chunks.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count
0,-41,Skip to content [image] Menu Primary Navigatio...,93,22,23.25
1,-40,Search Want to create or adapt books like this?,47,9,11.75
2,-40,Learn more about how Pressbooks supports open ...,67,9,16.75
3,-40,Book Title: Human Nutrition:,28,4,7.00
4,-39,2020 Edition by University of Hawai‘i at Mānoa...,384,68,96.00


Splitting each chunk into its own item

In [14]:
import re
pages_and_chunks=[]
for item in tqdm(pages_and_texts):
  for sentence_chunk in item["sentences"]:
    chunk_dict = {}
    chunk_dict["page_number"] = item["page_number"]


    join_sentence_chunk= "".join(sentence_chunk).replace(" ", " ").strip()
    join_sentence_chunk= re.sub(r'\.([A-Z])', r'. \1', join_sentence_chunk)
    chunk_dict["sentence_chunk"] = join_sentence_chunk

    #get stats about chunk
    chunk_dict["chunk_char_count"] = len(join_sentence_chunk)
    chunk_dict["chunk_word_count"] = len(join_sentence_chunk.split(" ")) # Corrected this line
    chunk_dict["chunk_token_count"] = len(join_sentence_chunk) / 4

    pages_and_chunks.append(chunk_dict)

len(pages_and_chunks)

  0%|          | 0/32 [00:00<?, ?it/s]

224

In [16]:
random.sample(pages_and_texts, k=1)

[{'page_number': -37,
  'page_char_count': 83,
  'page_word_count': 12,
  'page_sentence_count_raw': 1,
  'page_token_count': 20.75,
  'text': 'Creative Commons Attribution Read Book Contents Show All Contents Hide All Contents',
  'sentences': ['Creative Commons Attribution Read Book Contents Show All Contents Hide All Contents'],
  'page_sentence_count_spacy': 1,
  'sentences_chunks': [['Creative Commons Attribution Read Book Contents Show All Contents Hide All Contents']],
  'num_chunks': 1}]

In [17]:
df= pd.DataFrame(pages_and_chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,224.00,224.00,224.00,224.00
mean,-26.32,87.14,13.35,21.78
std,6.57,68.16,10.16,17.04
min,-41.00,2.00,1.00,0.50
25%,-32.00,13.00,2.00,3.25
50%,-26.00,112.00,17.00,28.00
75%,-21.00,124.25,19.00,31.06
max,-10.00,482.00,68.00,120.50


In [18]:
#show random chunk with under 30 tokens
min_token_length= 30
for row in df[df['chunk_token_count'] <= min_token_length].sample(5).iterrows():
  print(f'Chunk token count: {row[1]["chunk_token_count"]} | Text: {row[1]["sentence_chunk"]}')

Chunk token count: 2.0 | Text: License:
Chunk token count: 29.5 | Text: Young Adulthood University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 3.
Chunk token count: 25.75 | Text: University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 11.
Chunk token count: 29.75 | Text: Sports Nutrition University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 6.
Chunk token count: 30.0 | Text: Weight Management University of Hawai‘i at Mānoa Food Science and Human Nutrition Program and Human Nutrition Program 4.


In [19]:
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_length].to_dict(orient="records")
pages_and_chunks_over_min_token_len[:2]

[{'page_number': -39,
  'sentence_chunk': "2020 Edition by University of Hawai‘i at Mānoa Food Science and Human Nutrition Program [image] Download this book •  EPUB •  Digital PDF •  Print PDF •  Pressbooks XML Book Description: This textbook serves as an introduction to nutrition for undergraduate students and is the OER textbook for the FSHN 185 The Science of Human Nutrition course at the University of Hawai'i at Mānoa.",
  'chunk_char_count': 384,
  'chunk_word_count': 68,
  'chunk_token_count': 96.0},
 {'page_number': -39,
  'sentence_chunk': 'The book covers basic concepts in human nutrition, key information about essential nutrients, basic nutritional assessment, and nutrition across the lifespan.',
  'chunk_char_count': 158,
  'chunk_word_count': 21,
  'chunk_token_count': 39.5}]

## Embedding Model - mpnet-base-v2

In [20]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-mpnet-base-v2", device="cpu")
sentences = [
    "The Sentence Tranform Library provides an easy and open-source way to create embeddings.",
    "Sentences can be embedded one by one or as alist of strings"
]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
embeddings= embedding_model.encode(sentences)
embeddings_dict= dict(zip(sentences, embeddings))

for sentence, embedding in embeddings_dict.items():
  print(f"Sentence: {sentence}")
  print(f"Embedding: {embedding}")
  print("")

Sentence: The Sentence Tranform Library provides an easy and open-source way to create embeddings.
Embedding: [-3.33107635e-02  4.89020795e-02 -2.16649193e-02  4.65952009e-02
 -3.72122750e-02 -1.99266169e-02  1.00682490e-02 -6.64003268e-02
 -2.82911886e-03 -9.20438487e-03  3.15448791e-02  3.42672281e-02
 -3.02754920e-02  2.91308817e-02  4.13112305e-02 -6.44592196e-02
  5.46959005e-02  5.94419613e-03 -1.63708441e-02  1.43992975e-02
  4.85667884e-02  4.46756296e-02  3.45884473e-03  4.01629023e-02
 -3.49699636e-03 -4.97740842e-02 -2.05847877e-03 -1.86864287e-02
  5.75393885e-02 -2.26149568e-03 -2.61267275e-02  3.40121775e-03
  4.28933837e-02  5.47245052e-03  9.28902011e-07 -4.64756042e-03
 -3.10724601e-02 -6.86854450e-03  5.59972273e-03  1.56452935e-02
  5.06103821e-02 -6.43509999e-02  4.42699762e-03  4.33632992e-02
 -3.12533416e-02 -9.91237257e-03  4.14486229e-02  1.24321701e-02
  9.18519050e-02  6.56128451e-02 -1.87841877e-02 -1.55328913e-02
 -2.53281835e-03 -1.84446052e-02 -1.93161629e

In [22]:
from tqdm.auto import tqdm
embedding_model.to("cpu")

## Embed each chunk one by one
for item in tqdm(pages_and_chunks_over_min_token_len):
  item["embedding"] = embedding_model.encode(item["sentence_chunk"])

  0%|          | 0/68 [00:00<?, ?it/s]

In [24]:
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]


In [25]:
text_chunk_embeddings= embedding_model.encode(text_chunks,
                                              batch_size=32,
                                              convert_to_tensor=True)
text_chunk_embeddings

tensor([[ 0.0356,  0.0114, -0.0086,  ..., -0.0059, -0.0095,  0.0303],
        [ 0.0153,  0.0223,  0.0013,  ..., -0.0032, -0.0061, -0.0181],
        [ 0.0371,  0.0343, -0.0160,  ..., -0.0008,  0.0013,  0.0198],
        ...,
        [ 0.0283,  0.0685, -0.0083,  ..., -0.0157,  0.0207, -0.0172],
        [ 0.0435,  0.0134, -0.0119,  ..., -0.0252,  0.0286,  0.0077],
        [ 0.0472,  0.0474,  0.0010,  ..., -0.0244, -0.0135,  0.0012]])

In [27]:
from sentence_transformers import util, SentenceTransformer

embedding_model = SentenceTransformer("all-mpnet-base-v2", device="cpu")

## Retrieval

In [31]:
import torch
query= "micronutrients functions"
print(f"Query: {query}")

query_embedding = embedding_model.encode(query, convert_to_tensor=True)
from time  import perf_counter as timer

start_timer= timer()
dot_scores= util.dot_score(a= query_embedding,b= text_chunk_embeddings)[0]
end_timer= timer()

print(f" Time taken to get scores on {len(text_chunk_embeddings)} embeddings: {end_timer - start_timer: .5f} seconds.")

top_results_dot_product= torch.topk(dot_scores, k=5)
top_results_dot_product

Query: micronutrients functions
 Time taken to get scores on 68 embeddings:  0.00020 seconds.


torch.return_types.topk(
values=tensor([0.5302, 0.5217, 0.4850, 0.4811, 0.4644]),
indices=tensor([ 1, 23, 26, 64, 20]))